# Labeled Baseline: A/B/C/U 라벨 기반 RAG

챗봇 구조는 `MedicalBaseline.ipynb`(URL baseline)과 같습니다.

```
사용자 질문 → 의료 DB에서 검색 (BM25 + Dense → RRF → Rerank) → rerank 점수가 낮으면 거절 → Qwen이 검색 문서로 답변
```

`QueryClassification.ipynb`에서 만든 질문 라벨(A 의료 / B 의료 인접 / C 비의료 / U 불확실)을 아래 세 곳에 씁니다.

| | URL baseline | Labeled | Labeled + DB 보강 |
|---|---|---|---|
| 의료 DB | 의료 사이트 문서 | 의료 사이트 문서 | 의료 사이트 문서 **+ train 질문 중 라벨 A/B인 질문의 문서** |
| 거절 임계값 튜닝 세트 | URL 기준 | 라벨 기준 | 라벨 기준 |
| 평가 세트 | 라벨 기준 (세 시스템 공통) | ← | ← |

**DB 보강**: URL 기준 DB에는 일반 사이트(위키백과 등)에 있는 의료 문서가 빠져 있습니다. **train** 질문 중 라벨이 A/B인 질문에 딸린 문서를 사이트와 관계없이 DB에 추가합니다. RAG에서는 DB를 질문보다 먼저 만들어 두므로, 시험(validation) 질문의 라벨은 DB를 만드는 데 쓰지 않습니다. 실제 서비스에서 지난 의료 질문으로 DB를 늘려 가는 것과 같습니다.

**비교 방법**: 세 시스템을 같은 라벨 기반 평가 세트에서 평가합니다. 임계값은 각자의 튜닝 세트와 DB에서 정합니다. URL baseline이 원래 세트에서 보고한 수치도 참고용으로 함께 보여줍니다.

**사전 조건**: `QueryClassification.ipynb`를 실행해 `data/labels/query_labels.parquet`가 있어야 합니다. 런타임은 A100 또는 L4입니다.

> 정답이 일반 사이트에만 있는 시험 질문(`lab_med_ext_test`)은 대부분 정답 passage가 DB에 없습니다. 그래서 이 세트는 검색 점수 대신, DB의 다른 문서로 답하는지와 거절하는지를 참고로 봅니다.

## 0. 환경 설정

In [ ]:
%pip install -q -U datasets bm25s PyStemmer faiss-cpu sentence-transformers rouge-score accelerate

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os, re, json, random, hashlib
from pathlib import Path
from collections import defaultdict
from urllib.parse import urlparse

import numpy as np
import pandas as pd

# ===== 경로 =====
BASE_DIR = Path(os.environ.get("RAG_BASE_DIR", "/content/drive/MyDrive/RAG-Chat"))
DATA_DIR = BASE_DIR / "data" / "processed"
INDEX_DIR = BASE_DIR / "index"
RESULT_DIR = BASE_DIR / "results" / "labeled"
URL_RESULT_DIR = BASE_DIR / "results" / "baseline"     # MedicalBaseline 결과 (참고용)
LABEL_PATH = BASE_DIR / "data" / "labels" / "query_labels.parquet"
for d in (DATA_DIR, INDEX_DIR, RESULT_DIR):
    d.mkdir(parents=True, exist_ok=True)

# ===== 실험 설정 =====
SEED = 42
DATASET_NAME, DATASET_VERSION = "microsoft/ms_marco", "v1.1"

N_OOD_TEST = 1000      # validation 도메인 밖 질문 수
N_TUNE = 500           # train 튜닝 세트 (의료 / 도메인 밖 각각)

EMBED_MODEL = "BAAI/bge-base-en-v1.5"
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "  # bge 검색용 쿼리 instruction
RERANK_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
GEN_MODEL = "Qwen/Qwen2.5-7B-Instruct"

K_CAND = 100           # BM25 / Dense 후보 수
K_RERANK = 50          # Hybrid 상위 몇 개를 rerank할지
K_EVAL = 10            # Recall@10, MRR@10
K_CTX = 5              # LLM에 넣을 passage 수
RRF_K = 60
MAX_NEW_TOKENS = 128
GEN_BATCH = 16
GEN_CONFIGS = ["No-Retrieval", "Hybrid+Rerank"]   # 다른 검색 방식도 비교하려면 추가

REFUSAL = "I cannot answer this question based on the provided documents."

random.seed(SEED)
np.random.seed(SEED)

## 1. 데이터 준비 (MedicalBaseline과 동일)

In [ ]:
MEDICAL_DOMAINS = [
    "webmd.com", "webmd.boots.com", "mayoclinic.org", "mayoclinic.com", "medicinenet.com",
    "healthline.com", "medical-dictionary.thefreedictionary.com", "nih.gov", "medlineplus.gov",
    "cdc.gov", "everydayhealth.com", "nhs.uk", "medicalnewstoday.com", "rxlist.com",
    "medscape.com", "emedicinehealth.com", "wisegeekhealth.com", "health.howstuffworks.com",
    "cancer.org", "clevelandclinic.org", "patient.info", "hopkinsmedicine.org", "healthtap.com",
    "news-medical.net", "cancerresearchuk.org", "americanpregnancy.org", "spine-health.com",
    "who.int", "cancer.gov", "womenshealth.gov", "md-health.com", "heart.org", "healthhype.com",
    "familydoctor.org", "goodrx.com", "medhelp.org", "symptomfind.com", "medilexicon.com",
    "psychcentral.com", "healthcommunities.com", "med-health.net", "diabetes.org",
    "themedicalquestions.com", "health.harvard.edu", "drugs.com", "cancercenter.com",
    "rightdiagnosis.com", "innerbody.com", "labtestsonline.org", "sharecare.com", "umm.edu",
    "kidshealth.org", "health.com", "healthguideinfo.com", "medicalhealthtests.com",
]

def get_host(url):
    host = (urlparse(url).hostname or "").lower()
    return host[4:] if host.startswith("www.") else host

def is_medical_host(host):
    return any(host == d or host.endswith("." + d) for d in MEDICAL_DOMAINS)

def normalize_text(text):
    return re.sub(r"\s+", " ", str(text)).strip()

def has_answer(answers):
    return any(
        str(a).strip() and str(a).strip().lower().rstrip(".") != "no answer present"
        for a in (answers if answers is not None else [])
    )

In [ ]:
from datasets import load_dataset

dataset = load_dataset(DATASET_NAME, name=DATASET_VERSION)
qa = {split: dataset[split].to_pandas() for split in ("train", "validation")}
for split, df in qa.items():
    print(split, len(df))

In [ ]:
rows = []
for split, df in qa.items():
    for qid, p in zip(df["query_id"], df["passages"]):
        for text, sel, url in zip(p["passage_text"], p["is_selected"], p["url"]):
            rows.append((split, int(qid), normalize_text(text), int(sel), url))

flat = pd.DataFrame(rows, columns=["split", "query_id", "text", "is_selected", "url"])
flat = flat[flat["text"] != ""]
flat["host"] = flat["url"].map(get_host)
host_is_med = {h: is_medical_host(h) for h in flat["host"].unique()}
flat["is_medical"] = flat["host"].map(host_is_med)

corpus = (
    flat[flat["is_medical"]]
    .drop_duplicates("text")[["text", "url", "host"]]
    .reset_index(drop=True)
)
corpus.insert(0, "pid", np.arange(len(corpus)))   # pid == 행 번호 (인덱스 위치와 동일)
text2pid = dict(zip(corpus["text"], corpus["pid"]))

print(f"전체 passage: {len(flat):,} / 고유 텍스트: {flat['text'].nunique():,}")
print(f"의료 코퍼스: {len(corpus):,}")
print(corpus["host"].value_counts().head(10).to_string())

### 1-1. URL 기준 세트 (MedicalBaseline과 동일 코드)
URL baseline의 임계값을 똑같이 정하기 위해 URL 튜닝 세트를 그대로 재현합니다.

In [ ]:
MEDICAL_KEYWORDS = [
    "disease", "disorder", "syndrome", "diabetes", "cancer", "asthma", "pneumonia", "infection",
    "arthritis", "stroke", "hypertension", "anemia", "symptom", "pain", "fever", "headache",
    "nausea", "vomiting", "dizziness", "fatigue", "swelling", "bleeding", "diagnosis", "diagnose",
    "blood test", "biopsy", "mri", "ct scan", "x-ray", "ultrasound", "medication", "medicine",
    "drug", "antibiotic", "insulin", "vaccine", "treatment", "therapy", "surgery", "prescription",
    "side effect", "heart", "brain", "kidney", "liver", "lung", "blood pressure", "hormone",
    "immune system", "pregnancy", "depression", "anxiety", "adhd", "mental health", "bipolar",
]
keyword_pattern = re.compile(
    r"(?<!\w)(?:" + "|".join(re.escape(k) for k in MEDICAL_KEYWORDS) + r")(?!\w)",
    flags=re.IGNORECASE,
)

def build_query_sets(split):
    df = qa[split]
    fs = flat[flat["split"] == split]
    gold = (
        fs[fs["is_medical"] & (fs["is_selected"] == 1)]
        .groupby("query_id")["text"]
        .apply(lambda ts: sorted({text2pid[t] for t in ts}))
    )
    any_med = set(fs.loc[fs["is_medical"], "query_id"])

    base = pd.DataFrame({
        "query_id": df["query_id"].astype(int),
        "query": df["query"].map(normalize_text),
        "query_type": df["query_type"],
        "answers": df["answers"].map(list),
    })
    answerable = df["answers"].map(has_answer)

    med = base[answerable & base["query_id"].isin(gold.index)].copy()
    med["gold_pids"] = med["query_id"].map(gold)

    ood = base[
        answerable
        & ~base["query_id"].isin(any_med)
        & ~base["query"].str.contains(keyword_pattern)
    ].copy()
    ood["gold_pids"] = [[] for _ in range(len(ood))]
    return med.reset_index(drop=True), ood.reset_index(drop=True)

med_val, ood_val = build_query_sets("validation")
med_train, ood_train = build_query_sets("train")

qsets = {
    "med_test": med_val,
    "ood_test": ood_val.sample(N_OOD_TEST, random_state=SEED).reset_index(drop=True),
    "med_tune": med_train.sample(N_TUNE, random_state=SEED).reset_index(drop=True),
    "ood_tune": ood_train.sample(N_TUNE, random_state=SEED).reset_index(drop=True),
}
for name, q in qsets.items():
    print(f"{name:9s} {len(q):5d}")
display(qsets["med_test"].head())

### 1-2. 라벨 기반 세트
| 세트 | 정의 |
|---|---|
| `lab_med_test` | validation `med_url` 중 A (전체). 정답이 의료 사이트에 있는 의료 질문 |
| `lab_med_ext_test` | validation `no_med` 중 A (전체). 정답이 일반 사이트에만 있는 의료 질문. 정답 passage가 DB에 없을 수 있어 답변과 거절만 참고로 봄 (1-3에서 생성) |
| `lab_b_test` | validation `med_url` 중 B (검색 성능만 참고) |
| `lab_ood_test` | validation `no_med` 중 C, 1,000개 샘플 |
| `lab_med_tune` / `lab_ood_tune` | train `med_url`의 A / train `no_med`의 C에서 각 500개 |
| `lab_med_ext_tune` | train `no_med` 중 A. 정답 passage가 보강 DB에 있음 (DB 보강 시스템의 임계값 튜닝에 추가) |

In [ ]:
labels = pd.read_parquet(LABEL_PATH)
url_sets = qsets   # 1-1에서 만든 URL 기준 세트
FLAG_COLS = ["label", "high_risk", "time_sensitive"]

def med_with_label(med_df, split, label):
    lab = labels[(labels["split"] == split) & (labels["pool"] == "med_url") & (labels["label"] == label)]
    return med_df.merge(lab[["query_id"] + FLAG_COLS], on="query_id").reset_index(drop=True)

def ood_with_label(split, n):
    lab = labels[(labels["split"] == split) & (labels["pool"] == "no_med") & (labels["label"] == "C")]
    out = lab[["query_id", "query", "query_type", "answers"] + FLAG_COLS].copy()
    out["answers"] = out["answers"].map(list)
    out["gold_pids"] = [[] for _ in range(len(out))]
    return out.sample(min(n, len(out)), random_state=SEED).reset_index(drop=True)

lab_med_train = med_with_label(med_train, "train", "A")
qsets = {
    "url_med_test": url_sets["med_test"],
    "url_med_tune": url_sets["med_tune"],
    "url_ood_tune": url_sets["ood_tune"],
    "lab_med_test": med_with_label(med_val, "validation", "A"),
    "lab_b_test": med_with_label(med_val, "validation", "B"),
    "lab_ood_test": ood_with_label("validation", N_OOD_TEST),
    "lab_med_tune": lab_med_train.sample(min(N_TUNE, len(lab_med_train)), random_state=SEED).reset_index(drop=True),
    "lab_ood_tune": ood_with_label("train", N_TUNE),
}
for name, q in qsets.items():
    print(f"{name:13s} {len(q):5d}")
print("\nvalidation med_url 라벨 분포:")
print(labels[(labels["split"] == "validation") & (labels["pool"] == "med_url")]["label"].value_counts().to_string())

### 1-3. 의료 DB 보강
**train** 질문 중 라벨이 A/B인 질문에 딸린 passage를 사이트와 관계없이 DB에 추가합니다. validation 질문은 시험 문제이므로 DB를 만드는 데 쓰지 않습니다.
기존 URL DB를 앞에 그대로 두고 뒤에 붙이므로 pid가 두 DB에서 같습니다. URL DB에서 검색할 때는 앞부분만 씁니다.

In [ ]:
# 시험(validation) 질문의 라벨은 DB에 반영하지 않음: DB는 질문보다 먼저 만들어지는 것이므로
med_keys = labels.loc[(labels["split"] == "train") & labels["label"].isin(["A", "B"]), ["split", "query_id"]]
in_med_q = pd.MultiIndex.from_frame(flat[["split", "query_id"]]).isin(list(med_keys.itertuples(index=False, name=None)))
extra = (
    flat[in_med_q & ~flat["text"].isin(text2pid)]
    .drop_duplicates("text")[["text", "url", "host"]]
    .reset_index(drop=True)
)
extra.insert(0, "pid", np.arange(len(corpus), len(corpus) + len(extra)))
aug_corpus = pd.concat([corpus, extra], ignore_index=True)   # 행 번호 == pid
aug_text2pid = dict(zip(aug_corpus["text"], aug_corpus["pid"]))

def ext_med_set(split, need_gold):
    """정답이 일반 사이트에만 있는 A 질문. 정답 passage가 보강 DB에 있으면 그 pid를 표시"""
    lab = labels[(labels["split"] == split) & (labels["pool"] == "no_med") & (labels["label"] == "A")]
    fs = flat[(flat["split"] == split) & (flat["is_selected"] == 1)]
    gold = fs.groupby("query_id")["text"].apply(lambda ts: sorted({aug_text2pid[t] for t in ts if t in aug_text2pid}))
    out = lab[["query_id", "query", "query_type", "answers"] + FLAG_COLS].copy()
    out["answers"] = out["answers"].map(list)
    out["gold_pids"] = out["query_id"].map(gold).map(lambda g: g if isinstance(g, list) else [])
    if need_gold:   # 임계값 튜닝용 train 질문은 정답 passage가 DB에 있어야 함
        out = out[out["gold_pids"].map(len) > 0]
    return out.reset_index(drop=True)

qsets["lab_med_ext_test"] = ext_med_set("validation", need_gold=False)
ext_tune = ext_med_set("train", need_gold=True)
qsets["lab_med_ext_tune"] = ext_tune.sample(min(N_TUNE, len(ext_tune)), random_state=SEED).reset_index(drop=True)

print(f"URL DB {len(corpus):,} → 보강 DB {len(aug_corpus):,} (+{len(extra):,})")
print(extra["host"].value_counts().head(10).to_string())
for name in ["lab_med_ext_test", "lab_med_ext_tune"]:
    print(f"{name:17s} {len(qsets[name]):5d}")
in_db = qsets["lab_med_ext_test"]["gold_pids"].map(len) > 0
print(f"lab_med_ext_test 중 정답 passage가 보강 DB에 있는 질문: {in_db.sum()} / {len(in_db)}")

In [ ]:
LAB_DATA_DIR = DATA_DIR / "labeled"
LAB_DATA_DIR.mkdir(parents=True, exist_ok=True)
for name, q in qsets.items():
    q.to_parquet(LAB_DATA_DIR / f"{name}.parquet", index=False)
print("saved to", LAB_DATA_DIR)

## 2. 검색 (방식은 MedicalBaseline과 동일, URL DB와 보강 DB 각각)

In [ ]:
import bm25s, Stemmer

stemmer = Stemmer.Stemmer("english")
N_URL = len(corpus)
passages = aug_corpus["text"].tolist()   # 앞 N_URL개 = URL DB, 전체 = 보강 DB (pid 공유)
DB_SIZE = {"url": N_URL, "aug": len(passages)}

def build_bm25(texts):
    index = bm25s.BM25()
    index.index(bm25s.tokenize(texts, stopwords="en", stemmer=stemmer))
    return index

bm25_index = {db: build_bm25(passages[:n]) for db, n in DB_SIZE.items()}

def bm25_search(queries, db="url", k=K_CAND):
    tokens = bm25s.tokenize(queries, stopwords="en", stemmer=stemmer, show_progress=False)
    ids, scores = bm25_index[db].retrieve(tokens, k=k, show_progress=False)
    return ids, scores

In [ ]:
import torch, faiss
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer(EMBED_MODEL, device="cuda")
embedder.half()

def encode_passages(texts, path):
    if path.exists():
        return np.load(path)
    emb = embedder.encode(
        texts, batch_size=256, normalize_embeddings=True,
        convert_to_numpy=True, show_progress_bar=True,
    ).astype("float32")
    np.save(path, emb)
    return emb

model_tag = EMBED_MODEL.split("/")[-1]
extra_tag = hashlib.md5("\n".join(passages[N_URL:]).encode()).hexdigest()[:8]   # 라벨이 바뀌면 캐시도 새로
corpus_emb = np.concatenate([
    encode_passages(passages[:N_URL], INDEX_DIR / f"{model_tag}_{N_URL}.npy"),   # MedicalBaseline과 같은 캐시
    encode_passages(passages[N_URL:], INDEX_DIR / f"{model_tag}_aug_{extra_tag}.npy"),
])

faiss_index = {}
for db, n in DB_SIZE.items():
    faiss_index[db] = faiss.IndexFlatIP(corpus_emb.shape[1])   # 정규화된 벡터의 내적 = 코사인 유사도
    faiss_index[db].add(corpus_emb[:n])

def dense_search(queries, db="url", k=K_CAND):
    q_emb = embedder.encode(
        [QUERY_PREFIX + q for q in queries], batch_size=256,
        normalize_embeddings=True, convert_to_numpy=True,
    ).astype("float32")
    scores, ids = faiss_index[db].search(q_emb, k)
    return ids, scores

print(corpus_emb.shape)

In [ ]:
def rrf_fuse(*id_lists_per_run, k=RRF_K, top=K_RERANK):
    """run마다 (n_query, K) 형태의 id 배열을 받아 질의별로 RRF로 합친 상위 top개 pid 리스트를 반환."""
    fused = []
    for per_query in zip(*id_lists_per_run):
        score = defaultdict(float)
        for ids in per_query:
            for rank, pid in enumerate(ids):
                score[int(pid)] += 1.0 / (k + rank + 1)
        fused.append(sorted(score, key=score.get, reverse=True)[:top])
    return fused

In [ ]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(RERANK_MODEL, device="cuda", max_length=512)

def rerank(queries, candidates):
    """candidates: 질의별 pid 리스트 → (정렬된 pid 리스트, 정렬된 점수 리스트)"""
    pairs = [(q, passages[pid]) for q, cand in zip(queries, candidates) for pid in cand]
    flat_scores = reranker.predict(pairs, batch_size=256, show_progress_bar=False)
    out_ids, out_scores, pos = [], [], 0
    for cand in candidates:
        s = np.asarray(flat_scores[pos:pos + len(cand)])
        pos += len(cand)
        order = np.argsort(-s)
        out_ids.append([cand[i] for i in order])
        out_scores.append(s[order].tolist())
    return out_ids, out_scores

In [ ]:
def run_retrieval(queries, db):
    bm25_ids, _ = bm25_search(queries, db)
    dense_ids, _ = dense_search(queries, db)
    hybrid_ids = rrf_fuse(bm25_ids, dense_ids)
    rr_ids, rr_scores = rerank(queries, hybrid_ids)
    return {
        "BM25": [list(map(int, x)) for x in bm25_ids],
        "Dense": [list(map(int, x)) for x in dense_ids],
        "Hybrid": hybrid_ids,
        "Hybrid+Rerank": rr_ids,
    }, rr_scores

runs = {"url": {}, "aug": {}}            # runs[db][qset][method] = 질의별 pid 리스트 (순위순)
rerank_scores = {"url": {}, "aug": {}}
for name, q in qsets.items():
    for db in (["url", "aug"] if name.startswith("lab_") else ["url"]):
        runs[db][name], rerank_scores[db][name] = run_retrieval(q["query"].tolist(), db)
    print(f"{name} done")

### 2-1. 검색 평가: URL 세트 vs 라벨 세트

In [ ]:
def retrieval_metrics(run, gold, k=K_EVAL):
    recall, mrr = [], []
    for ids, g in zip(run, gold):
        g = set(g)
        top = list(ids)[:k]
        hits = [r for r, pid in enumerate(top) if pid in g]
        recall.append(len(set(top) & g) / len(g))
        mrr.append(1.0 / (hits[0] + 1) if hits else 0.0)
    return {f"Recall@{k}": np.mean(recall), f"MRR@{k}": np.mean(mrr)}

tables = {}
# lab_med_ext_test는 정답 passage가 DB에 없는 질문이 대부분이라 검색 점수에서 제외
for db, name in [("url", "url_med_test"), ("url", "lab_med_test"), ("url", "lab_b_test"),
                 ("aug", "lab_med_test"), ("aug", "lab_b_test")]:
    gold = qsets[name]["gold_pids"].tolist()
    tables[(db, name)] = pd.DataFrame({m: retrieval_metrics(r, gold) for m, r in runs[db][name].items()}).T
retrieval_table = pd.concat(tables, axis=1).round(4)
retrieval_table.to_csv(RESULT_DIR / "retrieval_metrics.csv")
retrieval_table

## 3. 임계값 (각 시스템의 튜닝 세트에서 따로 결정)

In [ ]:
def top_scores(name, db="url"):
    return np.array([s[0] for s in rerank_scores[db][name]])

def tune_threshold(med_names, ood_name, db="url"):
    med = np.concatenate([top_scores(n, db) for n in med_names])
    ood = top_scores(ood_name, db)
    cand = np.unique(np.concatenate([med, ood]))
    bal = [((med >= t).mean() + (ood < t).mean()) / 2 for t in cand]
    return float(cand[int(np.argmax(bal))]), max(bal)

THR_URL, bal_url = tune_threshold(["url_med_tune"], "url_ood_tune")
THR_LAB, bal_lab = tune_threshold(["lab_med_tune"], "lab_ood_tune")
THR_AUG, bal_aug = tune_threshold(["lab_med_tune", "lab_med_ext_tune"], "lab_ood_tune", db="aug")
print(f"THR_URL = {THR_URL:.4f} (tune bal acc {bal_url:.4f})")
print(f"THR_LAB = {THR_LAB:.4f} (tune bal acc {bal_lab:.4f})")
print(f"THR_AUG = {THR_AUG:.4f} (tune bal acc {bal_aug:.4f})")

### 3-1. 최종 DB 저장
챗봇 앱이 노트북 없이 바로 검색할 수 있도록 **Labeled + DB 보강** 시스템의 DB를 파일로 저장합니다 (`index/medical_db/`).

| 파일 | 내용 |
|---|---|
| `corpus.parquet` | pid, text, url, host (행 번호 == pid) |
| `faiss.index` | bge 임베딩 벡터 DB (내적 = 코사인 유사도) |
| `bm25/` | BM25 인덱스 |
| `config.json` | 모델 이름, 검색 설정, 거절 임계값 |

앱에서 불러오기:
```python
corpus = pd.read_parquet(DB_DIR / "corpus.parquet")
faiss_index = faiss.read_index(str(DB_DIR / "faiss.index"))
bm25 = bm25s.BM25.load(str(DB_DIR / "bm25"))
config = json.loads((DB_DIR / "config.json").read_text())
```
질문 검색 시 BM25는 `config["bm25"]`와 같은 불용어·스테머로, Dense는 `config["query_prefix"]`를 붙여 인코딩합니다.

In [ ]:
DB_DIR = INDEX_DIR / "medical_db"
DB_DIR.mkdir(parents=True, exist_ok=True)

aug_corpus.to_parquet(DB_DIR / "corpus.parquet", index=False)
faiss.write_index(faiss_index["aug"], str(DB_DIR / "faiss.index"))
bm25_index["aug"].save(str(DB_DIR / "bm25"))
db_config = {
    "n_docs": DB_SIZE["aug"],
    "n_url_docs": N_URL,             # 앞 n_url_docs개는 URL 기준 의료 사이트 문서, 나머지는 train 라벨로 보강한 문서
    "embed_model": EMBED_MODEL,
    "query_prefix": QUERY_PREFIX,
    "rerank_model": RERANK_MODEL,
    "bm25": {"stopwords": "en", "stemmer": "english"},
    "k_cand": K_CAND, "k_rerank": K_RERANK, "k_ctx": K_CTX, "rrf_k": RRF_K,
    "threshold": THR_AUG,            # rerank 1등 점수가 이보다 낮으면 거절
    "refusal": REFUSAL,
}
(DB_DIR / "config.json").write_text(json.dumps(db_config, indent=2, ensure_ascii=False))
print("saved to", DB_DIR)
for p in sorted(DB_DIR.iterdir()):
    size = sum(f.stat().st_size for f in p.rglob("*")) if p.is_dir() else p.stat().st_size
    print(f"  {p.name:15s} {size / 2**20:7.1f} MB")

## 4. 생성 모델

In [ ]:
SYSTEM_RAG = f"""You are a medical information assistant. Answer the question using ONLY the information in the provided passages.
Rules:
1. Do not use any knowledge that is not stated in the passages.
2. Answer in one or two complete sentences that restate the subject of the question.
3. If the passages do not contain the answer, reply exactly: {REFUSAL}"""

SYSTEM_CLOSED = "You are a medical information assistant. Answer the question in one or two complete sentences."

def build_messages(query, pids=None):
    if pids is None:
        return [{"role": "system", "content": SYSTEM_CLOSED},
                {"role": "user", "content": f"Question: {query}\nAnswer:"}]
    ctx = "\n\n".join(f"[{i}] {passages[pid]}" for i, pid in enumerate(pids, 1))
    return [{"role": "system", "content": SYSTEM_RAG},
            {"role": "user", "content": f"Passages:\n{ctx}\n\nQuestion: {query}\nAnswer:"}]

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm.auto import tqdm

tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL, padding_side="left")
llm = AutoModelForCausalLM.from_pretrained(GEN_MODEL, torch_dtype=torch.bfloat16, device_map="cuda")
llm.eval()

@torch.no_grad()
def generate(messages_list, batch_size=GEN_BATCH):
    prompts = [tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=True)
               for m in messages_list]
    order = np.argsort([len(p) for p in prompts])          # 길이순 배치로 padding 낭비 줄이기
    outputs = [None] * len(prompts)
    for i in tqdm(range(0, len(prompts), batch_size)):
        idx = order[i:i + batch_size]
        enc = tokenizer([prompts[j] for j in idx], return_tensors="pt", padding=True).to(llm.device)
        out = llm.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                           temperature=None, top_p=None, top_k=None)
        texts = tokenizer.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
        for j, t in zip(idx, texts):
            outputs[j] = t.strip()
    return outputs

In [ ]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

def _tokens(s):
    return re.findall(r"[a-z0-9]+", s.lower())

def is_refusal(pred):
    return "cannot answer" in pred.lower()

def answer_recall(pred, ref):
    ref_t, pred_t = _tokens(ref), set(_tokens(pred))
    return sum(t in pred_t for t in ref_t) / len(ref_t) if ref_t else 0.0

def generation_metrics(preds, answers_list):
    rouge = [max(scorer.score(a, p)["rougeL"].fmeasure for a in ans) for p, ans in zip(preds, answers_list)]
    recall = [max(answer_recall(p, a) for a in ans) for p, ans in zip(preds, answers_list)]
    return {"ROUGE-L": np.mean(rouge), "AnswerRecall": np.mean(recall),
            "RefusalRate": np.mean([is_refusal(p) for p in preds])}

## 5. 생성
- 의료 질문(`lab_med_test` + `lab_med_ext_test`): DB별로 게이트 없이 전체를 생성합니다. 시스템별 거절은 6장에서 마스크로 적용합니다.
- `lab_ood_test`: DB별로 그 DB를 쓰는 시스템 중 하나라도 통과시키는 질문만 생성합니다.

URL baseline과 Labeled는 같은 DB를 쓰므로 생성 결과를 공유합니다. 보강 DB에서도 검색 결과가 같으면 캐시된 생성 결과를 재사용합니다.

In [ ]:
MED_SETS = ["lab_med_test", "lab_med_ext_test"]   # 정답이 의료 사이트 / 일반 사이트에만
SYSTEMS = {   # 이름: (DB, 임계값)
    "URL baseline": ("url", THR_URL),
    "Labeled": ("url", THR_LAB),
    "Labeled + DB 보강": ("aug", THR_AUG),
}
mt = pd.concat([qsets[n].assign(subset=n) for n in MED_SETS], ignore_index=True)
ot = qsets["lab_ood_test"]

def med_runs(db):
    return [ids for n in MED_SETS for ids in runs[db][n]["Hybrid+Rerank"]]

def med_scores(db):
    return np.concatenate([top_scores(n, db) for n in MED_SETS])

_gen_cache = {}
def generate_cached(msgs):
    """질문과 passage가 같으면 생성 결과도 같으므로 한 번만 생성"""
    keys = [json.dumps(m) for m in msgs]
    todo = list({k: m for k, m in zip(keys, msgs) if k not in _gen_cache}.items())
    for (k, _), out in zip(todo, generate([m for _, m in todo])):
        _gen_cache[k] = out
    return [_gen_cache[k] for k in keys]

preds = {"No-Retrieval": generate([build_messages(q) for q in mt["query"]])}
for db in ("url", "aug"):
    preds[db] = generate_cached([build_messages(q, ids[:K_CTX]) for q, ids in zip(mt["query"], med_runs(db))])

gen_rows = {}
for cfg, p in preds.items():
    for n in MED_SETS:
        mask = mt["subset"].eq(n).to_numpy()
        gen_rows[(cfg, n)] = generation_metrics([x for x, k in zip(p, mask) if k], mt.loc[mask, "answers"].tolist())
gen_table = pd.DataFrame(gen_rows).T.round(4)
gen_table.to_csv(RESULT_DIR / "generation_metrics.csv")

ood_gen = {}
for db in ("url", "aug"):
    thr = min(t for d, t in SYSTEMS.values() if d == db)
    need = np.where(top_scores("lab_ood_test", db) >= thr)[0]
    out = [REFUSAL] * len(ot)
    msgs = [build_messages(ot["query"][i], runs[db]["lab_ood_test"]["Hybrid+Rerank"][i][:K_CTX]) for i in need]
    for i, a in zip(need, generate_cached(msgs)):
        out[i] = a
    ood_gen[db] = out
gen_table

## 6. 시스템 비교 (같은 라벨 기반 평가 세트)

| 지표 | 의미 |
|---|---|
| Refusal_medsite | 정답이 의료 사이트에 있는 A 질문을 거절한 비율. 정답 passage가 DB에 있으므로 거절은 오답 (낮을수록 좋음) |
| OODRefusal | C 질문을 거절한 비율 (높을수록 좋음) |
| BalancedAcc | (1 − Refusal_medsite + OODRefusal) / 2 |
| ROUGE-L / AnswerRecall_medsite | `lab_med_test` 기준. 거절한 답도 포함해 계산합니다 |
| Refusal_extsite / AnswerRecall_extsite | 정답이 일반 사이트에만 있는 A 질문 (참고). 정답 passage가 DB에 없을 수 있어 거절이 꼭 오답은 아닙니다. 답한 경우 정답 내용을 담았는지를 AnswerRecall로 봅니다 |

In [ ]:
def apply_gate(gen, passed):
    return [g if p else REFUSAL for g, p in zip(gen, passed)]

final, rows = {}, {}
for sys_name, (db, thr) in SYSTEMS.items():
    med_ans = apply_gate(preds[db], med_scores(db) >= thr)
    ood_ans = apply_gate(ood_gen[db], top_scores("lab_ood_test", db) >= thr)
    final[sys_name] = (med_ans, ood_ans)
    row = {}
    for n, tag in [("lab_med_test", "medsite"), ("lab_med_ext_test", "extsite")]:
        mask = mt["subset"].eq(n).to_numpy()
        m = generation_metrics([a for a, k in zip(med_ans, mask) if k], mt.loc[mask, "answers"].tolist())
        row[f"Refusal_{tag}"] = m["RefusalRate"]
        row[f"AnswerRecall_{tag}"] = m["AnswerRecall"]
        if tag == "medsite":
            row["ROUGE-L"] = m["ROUGE-L"]
    row["OODRefusal"] = np.mean([is_refusal(a) for a in ood_ans])
    row["BalancedAcc"] = (1 - row["Refusal_medsite"] + row["OODRefusal"]) / 2
    rows[sys_name] = row
cols = ["Refusal_medsite", "OODRefusal", "BalancedAcc", "ROUGE-L", "AnswerRecall_medsite",
        "Refusal_extsite", "AnswerRecall_extsite"]
compare_table = pd.DataFrame(rows).T[cols].round(4)
compare_table.to_csv(RESULT_DIR / "system_comparison.csv")
print(f"THR_URL = {THR_URL:.4f}, THR_LAB = {THR_LAB:.4f}, THR_AUG = {THR_AUG:.4f}")
compare_table

### 6-1. 참고: URL baseline이 자기 세트(URL 기준)에서 보고한 수치
`MedicalBaseline.ipynb`를 먼저 실행했다면 결과 CSV를 읽어 나란히 보여줍니다. 평가 세트 정의가 다르므로 직접 비교하지 말고, 세트 정의가 점수에 미치는 영향을 보는 용도로만 쓰세요.

In [ ]:
for fname in ["retrieval_metrics.csv", "final_pipeline.csv"]:
    path = URL_RESULT_DIR / fname
    if path.exists():
        print(f"== MedicalBaseline / {fname}")
        display(pd.read_csv(path, index_col=0))
    else:
        print("없음:", path)

### 6-2. 사례 저장

In [ ]:
cases = mt[["query_id", "subset", "query", "answers", "high_risk", "time_sensitive"]].copy()
ood_cases = ot[["query_id", "query", "answers"]].copy()
for db in ("url", "aug"):
    cases[f"top_score_{db}"] = med_scores(db)
    ood_cases[f"top_score_{db}"] = top_scores("lab_ood_test", db)
for sys_name, (med_ans, ood_ans) in final.items():
    cases[sys_name] = med_ans
    ood_cases[sys_name] = ood_ans
cases["No-Retrieval"] = preds["No-Retrieval"]
cases.to_json(RESULT_DIR / "lab_med_test_predictions.jsonl", orient="records", lines=True, force_ascii=False)
ood_cases.to_json(RESULT_DIR / "lab_ood_test_predictions.jsonl", orient="records", lines=True, force_ascii=False)

pd.set_option("display.max_colwidth", 200)
lab_sys, aug_sys = "Labeled", "Labeled + DB 보강"
print("DB 보강 후 새로 답하게 된 의료 질문:")
display(cases[cases[lab_sys].map(is_refusal) & ~cases[aug_sys].map(is_refusal)][["subset", "query", "answers", aug_sys]].head(10))
print("DB 보강 후 새로 통과된 OOD 질문 (부작용 확인):")
display(ood_cases[ood_cases[lab_sys].map(is_refusal) & ~ood_cases[aug_sys].map(is_refusal)][["query", aug_sys]].head(10))

## 7. 대화형 테스트

In [ ]:
def ask(query, db="aug", show_sources=True):
    thr = THR_AUG if db == "aug" else THR_LAB
    bm25_ids, _ = bm25_search([query], db)
    dense_ids, _ = dense_search([query], db)
    ids, scores = rerank([query], rrf_fuse(bm25_ids, dense_ids))
    ids, scores = ids[0][:K_CTX], scores[0][:K_CTX]
    if scores[0] < thr:
        answer, reason = REFUSAL, "low rerank score"
    else:
        answer, reason = generate([build_messages(query, ids)], batch_size=1)[0], "answered"
    print(f"Q: {query}\nA: {answer}\n   db={db} top={scores[0]:.3f} thr={thr:.3f} ({reason})")
    if show_sources and not is_refusal(answer):
        for i, (pid, s) in enumerate(zip(ids, scores), 1):
            print(f"  [{i}] ({s:.3f}) {aug_corpus.at[pid, 'url']}\n      {passages[pid][:150]}...")

ask("What are the symptoms of iron deficiency anemia?")
ask("What causes your hands to shake?")
ask("Who invented the MRI machine?")
ask("Who won the 2010 FIFA World Cup?")